In [2]:
import pandas as pd
sheets = pd.read_excel("./datasets/Scats Data October 2006.xls", sheet_name=None)
sheets.keys()

dict_keys(['Notes', 'Data', 'Summary Of Data'])

What does the notes have to say?

In [3]:
sheets["Notes"]["Unnamed: 1"].dropna().to_list()

['Request: ',
 'Attached is the 24 hour volume for 40 SCATS sites collected in Boroondara Council',
 'Collection Period: 1st October 2006 to 31st October 2006 (1st October - last Sunday of School holidays)',
 'Data is available every 15minutes for each day collected. Data was validated as per ISD standards for all sites.',
 'On selected days the Data for the following sites was very poor and was removed: ',
 970,
 2000,
 2820,
 3001,
 3002,
 3127,
 3682,
 3685,
 4057,
 4262,
 4263,
 4264,
 4272,
 4812]

In [4]:
removed_days = sheets["Notes"]["Unnamed: 1"].dropna().to_list()[5:]
removed_days = [int(i) for i in removed_days]
removed_days

[970,
 2000,
 2820,
 3001,
 3002,
 3127,
 3682,
 3685,
 4057,
 4262,
 4263,
 4264,
 4272,
 4812]

In [5]:
df_info = sheets["Summary Of Data"].loc[2:]
df_info.columns = df_info.iloc[0]
df_info = df_info[1:]
df_info["SCATS Number"] = df_info["SCATS Number"].ffill()

df_info.dropna(axis=1, inplace=True)
df_info = df_info.convert_dtypes().infer_objects()
df_info.head(15)

2,SCATS Number,Location,Total
3,0970,HIGH STREET_RD E of WARRIGAL_RD,31
4,0970,HIGH STREET_RD W of WARRIGAL_RD,30
5,0970,WARRIGAL_RD N of HIGH STREET_RD,31
6,0970,WARRIGAL_RD S of HIGH STREET_RD,31
7,2000,BURWOOD_HWY E of WARRIGAL_RD,29
8,2000,TOORAK_RD W of WARRIGAL_RD,29
9,2000,WARRIGAL_RD N of TOORAK_RD,28
10,2000,WARRIGAL_RD S of BURWOOD_HWY,28
11,2200,MAROONDAH_HWY E of UNION_RD,31
12,2200,MAROONDAH_HWY W of UNION_RD,31


Threshold of Days

In [6]:
df_info_filtered = df_info[df_info["Total"].between(7, 31)]
df_info_filtered

2,SCATS Number,Location,Total
3,0970,HIGH STREET_RD E of WARRIGAL_RD,31
4,0970,HIGH STREET_RD W of WARRIGAL_RD,30
5,0970,WARRIGAL_RD N of HIGH STREET_RD,31
6,0970,WARRIGAL_RD S of HIGH STREET_RD,31
7,2000,BURWOOD_HWY E of WARRIGAL_RD,29
...,...,...,...
137,4812,SWAN_ST SW of MADDEN_GV,26
138,4821,BURNLEY_ST S OF VICTORIA_ST,31
139,4821,VICTORIA_ST E OF BURNLEY_ST,31
140,4821,VICTORIA_ST W OF BURNLEY_ST,31


so these are locations with actually reasonable values, so we can filter out the rest of the shitfuckery

In [7]:

# imagine having good formatting
df = sheets["Data"]
df.columns = pd.Series(df.loc[0])
df = df.loc[1:]

# remove added time
df.loc[:, 'Date'] = pd.to_datetime(df['Date']).dt.date
df = df.convert_dtypes()

# filter good locations
df = df.loc[df["Location"].isin(df_info_filtered["Location"])]

# add location identifier: SCARS Number + VicRoads Internal
df.loc[:, "Identifier"] = df["SCATS Number"].astype(str) + " - " + df["HF VicRoads Internal"].astype(str)
df.columns = df.columns.str.strip()


df.head(10)


,SCATS Number,Location,CD_MELWAY,NB_LATITUDE,NB_LONGITUDE,HF VicRoads Internal,VR Internal Stat,VR Internal Loc,NB_TYPE_SURVEY,Date,...,V87,V88,V89,V90,V91,V92,V93,V94,V95,Identifier
1,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-01,...,97,97,66,81,50,59,47,29,34,0970 - 249
2,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-02,...,102,107,114,80,60,62,48,44,26,0970 - 249
3,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-03,...,132,114,86,93,90,73,57,29,40,0970 - 249
4,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-04,...,113,132,101,113,90,78,66,52,44,0970 - 249
5,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-05,...,120,116,113,99,91,61,55,49,36,0970 - 249
6,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-06,...,137,135,136,108,120,115,102,74,78,0970 - 249
7,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-07,...,153,133,109,140,121,123,115,107,127,0970 - 249
8,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-08,...,92,96,82,80,64,64,48,48,43,0970 - 249
9,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-09,...,107,111,98,83,69,57,50,25,36,0970 - 249
10,0970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-10,...,131,124,111,112,103,66,50,49,36,0970 - 249


In [8]:
pd.Categorical(df["NB_TYPE_SURVEY"])

[1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1]
Length: 4128
Categories (1, Int64): [1]

In [9]:
test_unique_cols = ["SCATS Number","HF VicRoads Internal","VR Internal Stat","VR Internal Loc","Location"]

for col in test_unique_cols:
    print(pd.Categorical(df.groupby("Identifier")[col].nunique()))


[1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1]
Length: 137
Categories (1, int64): [1]
[1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1]
Length: 137
Categories (1, int64): [1]
[1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1]
Length: 137
Categories (1, int64): [1]
[1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1]
Length: 137
Categories (1, int64): [1]
[1, 1, 1, 1, 1, ..., 1, 1, 1, 1, 1]
Length: 137
Categories (1, int64): [1]


Using Identifiers to identify groups. Now, we need to drop irrelevant group info:

- SCATS Number, VicRoads Internal (included in identifier)
- Location (can be accessible via identifier later)
- Latitude and Longtitude (accessible via Location) (wait keeping this is better)
- VR Internal Stat,VR Internal Loc,NB_TYPE_SURVEY (what even are these?)

In [10]:
groups = df.groupby(by="Identifier")
for name, group in groups:
    # print("Name:", name)
    # print("Info:", group.info())
    gr = group.copy()
    gr = gr.drop(columns=["SCATS Number", "HF VicRoads Internal", "Location", "VR Internal Stat", "VR Internal Loc", "NB_TYPE_SURVEY", "CD_MELWAY"])
    gr.reset_index(drop=True, inplace=True)
    print(gr.head(5))
    gr.to_csv(f"datasets/boroondara/{name}.csv")
    print(f"Saved Group {name} to boroondara/{name}.csv")

0  NB_LATITUDE  NB_LONGITUDE        Date  V00  V01  V02  V03  V04  V05  V06  \
0     -37.8676     145.09146  2006-10-01   92   93   90   55   64   50   47   
1     -37.8676     145.09146  2006-10-02   31   19   20   20   12    8   12   
2     -37.8676     145.09146  2006-10-03   20   29   16    9    8   13    8   
3     -37.8676     145.09146  2006-10-04   32   30   26   13   22   17   13   
4     -37.8676     145.09146  2006-10-05   42   37   19   18   12   20   23   

0  ...  V87  V88  V89  V90  V91  V92  V93  V94  V95    Identifier  
0  ...  117   87   82   55   58   57   51   41   41  0970 - 10503  
1  ...  107  107   88   66   54   68   58   49   29  0970 - 10503  
2  ...  144  100  100  102   86   85   68   53   37  0970 - 10503  
3  ...  114  131   94   84   82   90   76   50   45  0970 - 10503  
4  ...  158  127  118   92  105  113   78   64   62  0970 - 10503  

[5 rows x 100 columns]
Saved Group 0970 - 10503 to boroondara/0970 - 10503.csv
0  NB_LATITUDE  NB_LONGITUDE        D

Now we transform to the type used by the *other* model

In [11]:
times_dict = {}
vcols = [f"V{str(i).zfill(2)}" for i in range(96)]
for col in vcols:
    i = int(col[1:]) if len(col[1:]) > 0 else 0
    t = i * 15 
    h = t // 60 
    m = t % 60
    times_dict[col] = f"{str(h).zfill(2)}:{str(m).zfill(2)}"
times_dict

{'V00': '00:00',
 'V01': '00:15',
 'V02': '00:30',
 'V03': '00:45',
 'V04': '01:00',
 'V05': '01:15',
 'V06': '01:30',
 'V07': '01:45',
 'V08': '02:00',
 'V09': '02:15',
 'V10': '02:30',
 'V11': '02:45',
 'V12': '03:00',
 'V13': '03:15',
 'V14': '03:30',
 'V15': '03:45',
 'V16': '04:00',
 'V17': '04:15',
 'V18': '04:30',
 'V19': '04:45',
 'V20': '05:00',
 'V21': '05:15',
 'V22': '05:30',
 'V23': '05:45',
 'V24': '06:00',
 'V25': '06:15',
 'V26': '06:30',
 'V27': '06:45',
 'V28': '07:00',
 'V29': '07:15',
 'V30': '07:30',
 'V31': '07:45',
 'V32': '08:00',
 'V33': '08:15',
 'V34': '08:30',
 'V35': '08:45',
 'V36': '09:00',
 'V37': '09:15',
 'V38': '09:30',
 'V39': '09:45',
 'V40': '10:00',
 'V41': '10:15',
 'V42': '10:30',
 'V43': '10:45',
 'V44': '11:00',
 'V45': '11:15',
 'V46': '11:30',
 'V47': '11:45',
 'V48': '12:00',
 'V49': '12:15',
 'V50': '12:30',
 'V51': '12:45',
 'V52': '13:00',
 'V53': '13:15',
 'V54': '13:30',
 'V55': '13:45',
 'V56': '14:00',
 'V57': '14:15',
 'V58': '14:30

In [12]:
df[vcols].to_numpy(dtype=int)

array([[ 86,  83,  52, ...,  47,  29,  34],
       [ 32,  28,  17, ...,  48,  44,  26],
       [ 26,  32,  21, ...,  57,  29,  40],
       ...,
       [100,  81,  89, ...,  60,  49,  45],
       [ 40,  29,  36, ...,  62,  50,  62],
       [ 36,  30,  24, ...,  63,  51,  54]], shape=(4128, 96))

In [30]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import numpy as np
from typing import Any, Optional

def xiaochus_pipeline(identifier: str,
                      lags: int = 12,
                      train_ratio: float = 0.8,
                      seed: Optional[int] = None) -> Any:
#tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, MinMaxScaler]:
    df_id = pd.read_csv(f"datasets/boroondara/{identifier}.csv")
    flow = df_id[vcols].to_numpy(dtype=float).flatten()
    flow = np.array(np.nan_to_num(flow))
    cut = int(len(flow) * train_ratio)
    flow_train_raw = flow[:cut]
    flow_test_raw  = flow[cut:]
    
    scaler = MinMaxScaler(feature_range=(0, 1)).fit(flow_train_raw.reshape(-1, 1))

    flow_train = scaler.transform(flow_train_raw.reshape(-1, 1)).reshape(1, -1)[0]
    flow_test  = scaler.transform(flow_test_raw.reshape(-1, 1)).reshape(1, -1)[0]
    print(flow_train.shape)
    print(flow_test.shape)
    
    train_list, test_list = [], []
    for i in range(lags, len(flow_train)):
        train_list.append(flow_train[i - lags: i + 1])
    for i in range(lags, len(flow_test)):
        test_list.append(flow_test[i - lags: i + 1])
        
    train = np.array(train_list)
    test = np.array(test_list)
    print(f"Train:", train)
    print(f"Test:", test)
    print("Train shape:", train.shape)
    print("Test shape:", test.shape)
        
    if seed is not None:
        np.random.default_rng(seed)
        np.random.shuffle(train)
        
    X_train = train[:, :-1]
    y_train = train[:, -1]
    X_test = test[:, :-1]
    y_test = test[:, -1]
    print("Shape of X_train:", X_train.shape)
    print("Shape of X_test:", X_test.shape)
    print("Shape of y_train:", y_train.shape)
    print("Shape of y_test:", y_test.shape)
    
    return X_train, y_train, X_test, y_test, scaler


lag = 12    
X_train, X_test, y_train, y_test, scaler = xiaochus_pipeline("0970 - 249", lags=lag)
# print("Shape of X_train:", X_train.shape)
# print("Shape of X_test:", X_test.shape)
# print("Shape of y_train:", y_train.shape)
# print("Shape of y_test:", y_test.shape)
print(f"X_train = {X_train}")
print(f"y_train = {y_train}")
print(f"X_test = {X_test}")
print(f"y_test = {y_test}")
    

(2380,)
(596,)
Train: [[0.18930958 0.18262806 0.11358575 ... 0.03340757 0.05122494 0.05345212]
 [0.18262806 0.11358575 0.12694878 ... 0.05122494 0.05345212 0.05345212]
 [0.11358575 0.12694878 0.12917595 ... 0.05345212 0.05345212 0.0311804 ]
 ...
 [0.6481069  0.74387528 0.72160356 ... 0.78841871 0.73496659 0.6636971 ]
 [0.74387528 0.72160356 0.76391982 ... 0.73496659 0.6636971  0.60579065]
 [0.72160356 0.76391982 0.83296214 ... 0.6636971  0.60579065 0.53229399]]
Test: [[0.63697105 0.46547884 0.39198218 ... 0.27394209 0.28285078 0.29175947]
 [0.46547884 0.39198218 0.48552339 ... 0.28285078 0.29175947 0.28062361]
 [0.39198218 0.48552339 0.34521158 ... 0.29175947 0.28062361 0.19153675]
 ...
 [0.35857461 0.31625835 0.30734967 ... 0.17371938 0.155902   0.12026726]
 [0.31625835 0.30734967 0.3674833  ... 0.155902   0.12026726 0.09799555]
 [0.30734967 0.3674833  0.34521158 ... 0.12026726 0.09799555 0.07126949]]
Train shape: (2368, 13)
Test shape: (584, 13)
Shape of X_train: (2368, 12)
Shape of 

Loss Metric: MAPE

In [31]:
def MAPE(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean Absolute Percentage Error
    Calculate the mape.

    # Arguments
        y_true: List/ndarray, ture data.
        y_pred: List/ndarray, predicted data.
    # Returns
        mape: Double, result data for train.
    """

    y = [x for x in y_true if x > 0]
    y_pred = np.array([y_pred[i] for i in range(len(y_true)) if y_true[i] > 0])

    num: int = len(y_pred)
    sums: float = 0

    for i in range(num):
        tmp = abs(y[i] - y_pred[i]) / y[i]
        sums += tmp

    mape = sums * (100 / num)

    return mape

In [32]:
from model.model import get_cnn, get_gru, get_lstm, get_saes
from train import train_model, train_seas
from keras import Sequential



config: dict[str, Any] = {"batch": 256, "epochs": 600}

non_saes_layers = [lag, 64, 64, 1]
saes_layers = [lag, 400, 400, 400, 1]

models: dict[str, Sequential] = {
	"lstm": get_lstm(non_saes_layers), 
	"gru": get_gru(non_saes_layers),
	"cnn": get_cnn(non_saes_layers),
	"saes": get_saes(saes_layers)
}

def train_pipeline(identifier: str, model: str, lag: int = 12, config: dict[str, Any] = {"batch": 256, "epochs": 600}) -> None:
    X_train, y_train, _, _, _ = xiaochus_pipeline(identifier, lag)
    model = model.strip().lower()
    if model == "saes":
        X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
        m = models["saes"]
        train_seas(m, X_train, y_train, model, config)
    else:
        X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
        m = models[model]
        train_model(m, X_train, y_train, model, config)
    

identifier = "0970 - 249"

train_pipeline(identifier, "lstm", config={"batch": 256, "epochs": 10})


(2380,)
(596,)
Train: [[0.18930958 0.18262806 0.11358575 ... 0.03340757 0.05122494 0.05345212]
 [0.18262806 0.11358575 0.12694878 ... 0.05122494 0.05345212 0.05345212]
 [0.11358575 0.12694878 0.12917595 ... 0.05345212 0.05345212 0.0311804 ]
 ...
 [0.6481069  0.74387528 0.72160356 ... 0.78841871 0.73496659 0.6636971 ]
 [0.74387528 0.72160356 0.76391982 ... 0.73496659 0.6636971  0.60579065]
 [0.72160356 0.76391982 0.83296214 ... 0.6636971  0.60579065 0.53229399]]
Test: [[0.63697105 0.46547884 0.39198218 ... 0.27394209 0.28285078 0.29175947]
 [0.46547884 0.39198218 0.48552339 ... 0.28285078 0.29175947 0.28062361]
 [0.39198218 0.48552339 0.34521158 ... 0.29175947 0.28062361 0.19153675]
 ...
 [0.35857461 0.31625835 0.30734967 ... 0.17371938 0.155902   0.12026726]
 [0.31625835 0.30734967 0.3674833  ... 0.155902   0.12026726 0.09799555]
 [0.30734967 0.3674833  0.34521158 ... 0.12026726 0.09799555 0.07126949]]
Train shape: (2368, 13)
Test shape: (584, 13)
Shape of X_train: (2368, 12)
Shape of 

In [ ]:
from keras.saving import load_model
lstm = Sequential(load_model('model/lstm.keras'))
gru = Sequential(load_model('model/gru.keras'))
saes = Sequential(load_model('model/saes.keras'))
cnn = Sequential(load_model("model/cnn.keras"))
print(type(lstm))
print(type(gru))
print(type(saes))
print(type(cnn))

<class 'keras.src.models.sequential.Sequential'>
<class 'keras.src.models.sequential.Sequential'>
<class 'keras.src.models.sequential.Sequential'>
<class 'keras.src.models.sequential.Sequential'>


In [20]:
from glob import glob
len(glob("datasets/boroondara/*.csv"))

137